In [1]:
import os
from cdo import Cdo

In [2]:
cdo = Cdo(cdo='/usr/local/apps/cdo/2.4.0/bin/cdo')

# Helper functions

In [3]:
def remap_files(input_folder, output_folder, file_filter):
    """
    Remap netCDF files to 1°x1° grid using nearest neighbor.
    """
    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(input_folder):
        if filename.endswith(".nc") and file_filter in filename:
            input_file = os.path.join(input_folder, filename)
            output_file = os.path.join(output_folder, f"remapped_{filename}")

            if not os.path.exists(output_file):  # Avoid reprocessing
                cdo.remapnn("r180x90", input=input_file, output=output_file)
                print(f"✅ Remapped: {filename}")
            else:
                print(f"⏩ Skipped (already exists): {filename}")

In [4]:
def extract_variables(input_folder, output_folder, variables,
                      filename_filter=None, exclude_filter=None,
                      year_range=None):
    """
    Extract selected variables from remapped files.
    Optional year filtering supported.
    """
    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(input_folder):
        if not filename.endswith(".nc"):
            continue

        if filename_filter and filename_filter not in filename:
            continue

        if exclude_filter and exclude_filter in filename:
            continue

        # Optional year filter
        if year_range:
            try:
                year_str = filename.split("_")[-1].replace(".nc", "")
                year = int(year_str.split("-")[0])
                if not (year_range[0] <= year <= year_range[1]):
                    continue
            except:
                continue

        infile = os.path.join(input_folder, filename)
        outfile = os.path.join(output_folder, f"small_{filename}")

        if not os.path.exists(outfile):
            cdo.selname(variables, input=infile, output=outfile)
            print(f"✅ Extracted vars from {filename}")
        else:
            print(f"⏩ Skipped (already exists): {filename}")

# Experiments configuration

In [12]:
experiments = {
    #"exp3": "/lus/h2resw01/scratch/itcv/ece4/exp3/output/",
    #"XPPI": "/lus/h2resw01/scratch/ccpd/ece4/XPPI/output/",
    #"XEPI": "/lus/h2resw01/scratch/ccpd/ece4/XEPI/output/",
    "XE3C": "/lus/h2resw01/scratch/ccpd/ece4/XE3C/output/",
    #"XE6C": "/lus/h2resw01/scratch/ccpd/ece4/XE6C/output/",
}

In [9]:
experiments = {"pex3": "/lus/h2resw01/scratch/ecme3497/ece4/pex3/output/"}

In [13]:
base_output = "/lus/h2resw01/hpcperm/ecme3497/data-analysis/epochal/"

atm_vars = "tas,pr,rsut,rlut,rsdt"
oce_vars = "tos,sos"

# Main processing loop

In [14]:
for exp_name, input_base in experiments.items():

    print(f"\n================ {exp_name} ================\n")

    # -------------------------
    # Remap atmosphere (OIFS)
    # -------------------------
    remap_files(
        input_folder=os.path.join(input_base, "oifs"),
        output_folder=os.path.join(base_output, exp_name, "remaped/oifs"),
        file_filter="_1m_"
    )

    # -------------------------
    # Remap ocean (NEMO)
    # -------------------------
    remap_files(
        input_folder=os.path.join(input_base, "nemo"),
        output_folder=os.path.join(base_output, exp_name, "remaped/nemo"),
        file_filter="oce_1m_T"
    )

    # -------------------------
    # Extract ATM variables
    # -------------------------
    extract_variables(
        input_folder=os.path.join(base_output, exp_name, "remaped/oifs"),
        output_folder=os.path.join(base_output, exp_name, "variables/atm"),
        variables=atm_vars,
        filename_filter="atm_cmip6_1m",
        exclude_filter="_pl_"
    )

    # -------------------------
    # Extract OCE variables
    # -------------------------
    extract_variables(
        input_folder=os.path.join(base_output, exp_name, "remaped/nemo"),
        output_folder=os.path.join(base_output, exp_name, "variables/oce"),
        variables=oce_vars
    )


================ XE3C ================

⏩ Skipped (already exists): remapped_XE3C_atm_cmip6_1m_1819-1819.nc
⏩ Skipped (already exists): remapped_pex3_atm_cmip6_1m_1799-1799.nc
⏩ Skipped (already exists): remapped_XE3C_atm_cmip6_1m_1764-1764.nc
⏩ Skipped (already exists): remapped_XE3C_atm_cmip6_1m_1817-1817.nc
⏩ Skipped (already exists): remapped_pex3_atm_cmip6_1m_1726-1726.nc
⏩ Skipped (already exists): remapped_XE3C_atm_cmip6_1m_1880-1880.nc
⏩ Skipped (already exists): remapped_pex3_atm_cmip6_1m_1741-1741.nc
⏩ Skipped (already exists): remapped_pex3_atm_cmip6_1m_1704-1704.nc
⏩ Skipped (already exists): remapped_XE3C_atm_cmip6_1m_1702-1702.nc
⏩ Skipped (already exists): remapped_pex3_atm_cmip6_1m_1779-1779.nc
⏩ Skipped (already exists): remapped_pex3_atm_cmip6_1m_1739-1739.nc
⏩ Skipped (already exists): remapped_XE3C_atm_cmip6_1m_1834-1834.nc
⏩ Skipped (already exists): remapped_XE3C_atm_cmip6_1m_1883-1883.nc
⏩ Skipped (already exists): remapped_pex3_atm_cmip6_1m_1725-1725.nc
⏩ Skipp